# Clase 12: Entrenamiento de Agentes RL con PPO, DQN y Stable-Baselines3

**Machine Learning (RIA-805) — Bloque 2 — Factibilidad Técnico-Financiera**

**Docente:** Marco Capio  
**Fecha:** Semana 3 del bloque comprimido

**Objetivo:** Entrenar, evaluar y comparar agentes de RL usando Stable-Baselines3 con los algoritmos PPO y DQN en el entorno CartPole, documentando el aprendizaje mediante curvas de recompensa.

**Conexión con la práctica:** En P2-8 a P2-10 entrenarán y evaluarán agentes, y los compararán con controladores clásicos. Hoy verán exactamente cómo se hace.

## ¿Dónde estamos?

En la Clase 11 introdujimos el Aprendizaje por Refuerzo (RL) y Gymnasium. Vimos que un agente que elige acciones al azar apenas dura unos 20 pasos, en promedio. Hoy entrenaremos un agente real para que aprenda a resolver CartPole y compararemos dos algoritmos populares.

> **La diferencia clave:** en lugar de elegir acciones al azar, el agente usa la experiencia para mejorar su política.

## ¿Qué es un algoritmo de RL?

Un algoritmo de Aprendizaje por Refuerzo convierte la experiencia de interacción con el entorno en una política que busca maximizar la recompensa acumulada. En términos generales, combina tres componentes:

1. **Recolector de experiencia:** ejecuta acciones y registra transiciones como estado, acción, recompensa y siguiente estado.
2. **Estimador de valor o política:** estima qué tan conveniente es un estado/acción o directamente qué acción elegir.
3. **Mecanismo de actualización:** utiliza la experiencia recolectada para mejorar las estimaciones y la política.

Los algoritmos modernos usan redes neuronales para aproximar la política. El diseño de cada algoritmo determina cómo recolecta y reutiliza la experiencia, y cómo limita o estabiliza las actualizaciones.

## PPO: Proximal Policy Optimization

**PPO** es un algoritmo de gradiente de política que busca mejorar la política sin permitir cambios demasiado grandes en cada actualización. Su idea central es el *clipping*: limita la razón entre la probabilidad de una acción bajo la política nueva y la política anterior. Esto ayuda a evitar actualizaciones bruscas.

| Aspecto | Descripción |
|---|---|
| Tipo | *On-policy*: aprende principalmente de experiencia reciente de su política actual |
| Acciones | Discretas y continuas |
| Estabilidad | Alta en muchos problemas prácticos |
| Eficiencia de muestras | Media; no reutiliza la experiencia tantas veces como un método *off-policy* |

**Ventajas:** suele ser robusto y sencillo de aplicar, funciona con espacios de acción variados y ofrece un buen punto de partida.  
**Desventajas:** puede requerir muchas interacciones con el entorno y su desempeño depende de la configuración de los hiperparámetros.

**Hiperparámetros clave:**

- `learning_rate`: tamaño de los pasos de optimización.
- `n_steps`: pasos recolectados por entorno antes de actualizar la política.
- `batch_size`: ejemplos procesados en cada minibatch.
- `gamma`: factor de descuento de recompensas futuras.
- `clip_range`: límite de clipping que restringe cuánto puede cambiar la política.

## DQN: Deep Q-Network

**DQN** aproxima con una red neuronal la función de valor de acción `Q(s, a)`: el retorno esperado al ejecutar la acción `a` en el estado `s` y continuar después con la política aprendida. Selecciona la acción con mayor valor Q, mientras reserva oportunidades para explorar.

| Aspecto | Descripción |
|---|---|
| Tipo | *Off-policy*: puede aprender de transiciones almacenadas en experiencias anteriores |
| Acciones | Solo espacios discretos |
| Estabilidad | Media; la red objetivo y el buffer de repetición ayudan a estabilizar el aprendizaje |
| Eficiencia de muestras | Alta en comparación con métodos *on-policy*, pues reutiliza experiencias |

**Ventajas:** reutiliza datos mediante un *replay buffer* y puede ser eficiente en espacios de acciones discretas.  
**Desventajas:** no admite acciones continuas directamente y puede ser sensible a hiperparámetros y a la escala del problema.

**Hiperparámetros clave:**

- `learning_rate`: tamaño de los pasos de optimización.
- `buffer_size`: capacidad del buffer que almacena transiciones.
- `learning_starts`: interacciones mínimas antes de comenzar a actualizar la red.
- `batch_size`: cantidad de transiciones muestreadas por actualización.
- `gamma`: factor de descuento de recompensas futuras.
- `target_update_interval`: frecuencia con que se actualiza la red objetivo.

## ¿PPO o DQN? Guía de elección

| Criterio | PPO | DQN |
|---|---|---|
| Acciones | Discretas y continuas | Solo discretas |
| Estabilidad | Generalmente alta | Media; depende de la tarea y la configuración |
| Eficiencia de muestras | Media; datos recientes | Alta; reutiliza el buffer de experiencias |
| Facilidad de ajuste | Buen valor predeterminado para comenzar | Requiere atención a exploración, buffer y red objetivo |
| Uso recomendado | Acciones continuas o una primera solución robusta | Acciones discretas y necesidad de reutilizar experiencia |

**Regla práctica:** acciones continuas → PPO o SAC; acciones discretas → DQN o PPO; si hay dudas → PPO es un punto de partida razonable. La elección final debe validarse en el entorno y con varias semillas.

## Stable-Baselines3

**Stable-Baselines3 (SB3)** es una biblioteca de Python que ofrece implementaciones listas para usar de algoritmos de RL basados en PyTorch, con una API uniforme para entrenar y evaluar agentes.

Entre sus algoritmos están **PPO, A2C, DQN, SAC y TD3**. Para instalar la biblioteca en el entorno de Python activo:

```bash
pip install stable-baselines3
```

Sus ventajas incluyen una API simple, algoritmos probados y buena documentación. Para este ejemplo, CartPole se integra directamente con Gymnasium.

In [ ]:
# Importamos las herramientas para RL, análisis, visualización y manejo de archivos.
import os
import time
from pathlib import Path

import gymnasium as gym
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import stable_baselines3
from stable_baselines3 import DQN, PPO

# Activamos gráficos en línea y definimos un tamaño legible.
%matplotlib inline
plt.rcParams['figure.figsize'] = (10, 6)

# Detectamos si Jupyter se inició desde la raíz del proyecto o desde notebooks/.
current_dir = Path.cwd()
if (current_dir / 'notebooks').is_dir():
    PROJECT_ROOT = current_dir
elif current_dir.name == 'notebooks':
    PROJECT_ROOT = current_dir.parent
else:
    raise FileNotFoundError(
        'Abre este notebook desde la raíz del proyecto o desde su carpeta notebooks/.'
    )

# Creamos las carpetas de salida necesarias, sin sobrescribir archivos existentes.
MODELS_DIR = PROJECT_ROOT / 'models'
REPORTS_DIR = PROJECT_ROOT / 'reports'
FIGURES_DIR = REPORTS_DIR / 'figures'
for output_dir in (MODELS_DIR, REPORTS_DIR, FIGURES_DIR):
    output_dir.mkdir(parents=True, exist_ok=True)

print(f'Gymnasium: {gym.__version__}')
print(f'Stable-Baselines3: {stable_baselines3.__version__}')
print(f'Carpeta de resultados: {REPORTS_DIR}')

## Crear el entorno CartPole

Vamos a crear el entorno CartPole. Lo usaremos para entrenar nuestros agentes. El carro debe equilibrar un poste aplicando fuerza hacia la izquierda o la derecha; cada paso válido aporta una recompensa de 1 y el episodio termina si el poste cae o se alcanza el límite de tiempo.

In [ ]:
# Creamos la versión de CartPole con límite de 500 pasos por episodio.
env = gym.make('CartPole-v1')

# Inspeccionamos las acciones disponibles y las variables observadas.
print('Espacio de acciones:', env.action_space)
print('Espacio de observaciones:', env.observation_space)

## Línea base: agente aleatorio

Antes de entrenar, medimos la recompensa de un agente aleatorio. Esta es nuestra línea base. En cada paso selecciona una acción válida al azar; el resultado puede variar entre episodios, así que reportaremos varias ejecuciones y sus estadísticas.

In [ ]:
def evaluar_agente_aleatorio(env, n_episodios=10):
    """Evalúa una política aleatoria y devuelve la recompensa de cada episodio.

    Args:
        env: Entorno Gymnasium ya creado.
        n_episodios: Número de episodios completos que se ejecutarán.

    Returns:
        Lista de recompensas acumuladas, una por episodio.
    """
    recompensas = []
    env.action_space.seed(42)

    for episodio in range(n_episodios):
        observacion, info = env.reset(seed=42 + episodio)
        recompensa_total = 0.0
        terminado = False
        truncado = False

        while not (terminado or truncado):
            accion = env.action_space.sample()
            observacion, recompensa, terminado, truncado, info = env.step(accion)
            recompensa_total += recompensa

        recompensas.append(recompensa_total)

    return recompensas


# Ejecutamos la línea base y resumimos su distribución de recompensas.
recompensas_aleatorio = evaluar_agente_aleatorio(env, n_episodios=10)
print(f'Agente aleatorio — promedio: {np.mean(recompensas_aleatorio):.2f}')
print(f'Agente aleatorio — máximo: {np.max(recompensas_aleatorio):.2f}')
print(f'Agente aleatorio — mínimo: {np.min(recompensas_aleatorio):.2f}')

## Entrenar PPO

Ahora entrenamos un agente PPO. Usaremos `MlpPolicy` y 20,000 pasos de entrenamiento. **`MlpPolicy`** indica que la política y la función de valor usan una red neuronal tipo perceptrón multicapa, adecuada para las observaciones vectoriales de CartPole. **`total_timesteps`** es el número de interacciones con el entorno que el algoritmo utilizará para aprender; no equivale al número de episodios, pues cada episodio tiene una duración variable.

In [ ]:
# Fijamos una semilla para que la inicialización del modelo sea reproducible.
model_ppo = PPO('MlpPolicy', env, verbose=0, seed=42)

# Medimos el tiempo de entrenamiento para documentar el costo computacional.
inicio_ppo = time.perf_counter()
model_ppo.learn(total_timesteps=20000)
tiempo_ppo = time.perf_counter() - inicio_ppo
print(f'Tiempo de entrenamiento PPO: {tiempo_ppo:.2f} segundos')

# Guardamos el modelo entrenado en la carpeta de modelos del proyecto.
ruta_modelo_ppo = MODELS_DIR / 'ppo_cartpole.zip'
model_ppo.save(str(ruta_modelo_ppo))
print(f'Modelo PPO guardado en: {ruta_modelo_ppo}')

## Evaluar PPO

Evaluamos el agente PPO entrenado en 10 episodios. Comparamos con el agente aleatorio. La evaluación determinista selecciona la acción preferida por la política, sin añadir la exploración estocástica que puede usarse durante el aprendizaje.

In [ ]:
def evaluar_agente_entrenado(model, env, n_episodios=10):
    """Evalúa un modelo entrenado de SB3 en episodios independientes.

    Args:
        model: Modelo de Stable-Baselines3 con método predict.
        env: Entorno Gymnasium ya creado.
        n_episodios: Número de episodios completos que se ejecutarán.

    Returns:
        Lista de recompensas acumuladas, una por episodio.
    """
    recompensas = []

    for episodio in range(n_episodios):
        observacion, info = env.reset(seed=1000 + episodio)
        recompensa_total = 0.0
        terminado = False
        truncado = False

        while not (terminado or truncado):
            accion, estado = model.predict(observacion, deterministic=True)
            observacion, recompensa, terminado, truncado, info = env.step(accion)
            recompensa_total += recompensa

        recompensas.append(recompensa_total)

    return recompensas


# Calculamos métricas de PPO y las contrastamos con la línea base aleatoria.
recompensas_ppo = evaluar_agente_entrenado(model_ppo, env, n_episodios=10)
print(f'PPO — promedio: {np.mean(recompensas_ppo):.2f}')
print(f'PPO — máximo: {np.max(recompensas_ppo):.2f}')
print(f'PPO — mínimo: {np.min(recompensas_ppo):.2f}')
print(f'Promedio aleatorio para comparar: {np.mean(recompensas_aleatorio):.2f}')

## Curva de aprendizaje PPO

La curva de aprendizaje muestra cómo mejora la recompensa durante el entrenamiento. Si la curva sube, el agente aprende; si es plana, no mejora de forma apreciable. Cada punto representa la recompensa de un episodio registrado durante el aprendizaje. La media móvil ayuda a distinguir la tendencia de las fluctuaciones entre episodios.

In [ ]:
# Leemos las recompensas por episodio que SB3 conservó durante el entrenamiento.
recompensas_episodios_ppo = [float(info['r']) for info in model_ppo.ep_info_buffer]
if not recompensas_episodios_ppo:
    raise RuntimeError('PPO no registró episodios; no es posible graficar la curva.')

serie_ppo = pd.Series(recompensas_episodios_ppo, name='Recompensa por episodio')
media_movil_ppo = serie_ppo.rolling(window=50, min_periods=1).mean()

# Graficamos recompensa episódica y tendencia suavizada.
fig, ax = plt.subplots()
ax.plot(serie_ppo.index + 1, serie_ppo, alpha=0.45, label='Recompensa por episodio')
ax.plot(media_movil_ppo.index + 1, media_movil_ppo, linewidth=2, label='Media móvil (50 episodios)')
ax.set(title='Curva de aprendizaje — PPO', xlabel='Episodio', ylabel='Recompensa')
ax.legend()
ax.grid(alpha=0.25)
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'ppo_learning_curve.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Gráfico guardado en: {FIGURES_DIR / "ppo_learning_curve.png"}')

## Entrenar DQN

Ahora entrenamos un agente DQN con los mismos 20,000 pasos. DQN solo funciona con acciones discretas, y CartPole tiene acciones discretas: empujar el carro a la izquierda o a la derecha. Usar el mismo presupuesto de pasos facilita una comparación inicial, aunque una comparación rigurosa también probaría varias semillas y presupuestos.

In [ ]:
# Inicializamos DQN con la misma semilla y sin mostrar logs de entrenamiento.
model_dqn = DQN('MlpPolicy', env, verbose=0, seed=42)

# Medimos el tiempo de entrenamiento y aplicamos el mismo presupuesto de pasos.
inicio_dqn = time.perf_counter()
model_dqn.learn(total_timesteps=20000)
tiempo_dqn = time.perf_counter() - inicio_dqn
print(f'Tiempo de entrenamiento DQN: {tiempo_dqn:.2f} segundos')

# Guardamos el modelo entrenado en la carpeta de modelos del proyecto.
ruta_modelo_dqn = MODELS_DIR / 'dqn_cartpole.zip'
model_dqn.save(str(ruta_modelo_dqn))
print(f'Modelo DQN guardado en: {ruta_modelo_dqn}')

In [ ]:
# Evaluamos DQN con las mismas condiciones que PPO y el agente aleatorio.
recompensas_dqn = evaluar_agente_entrenado(model_dqn, env, n_episodios=10)
print(f'DQN — promedio: {np.mean(recompensas_dqn):.2f}')
print(f'DQN — máximo: {np.max(recompensas_dqn):.2f}')
print(f'DQN — mínimo: {np.min(recompensas_dqn):.2f}')
print(f'Promedio PPO: {np.mean(recompensas_ppo):.2f}')
print(f'Promedio aleatorio: {np.mean(recompensas_aleatorio):.2f}')

## Curva de aprendizaje DQN

Graficamos la curva de DQN para comparar con PPO. Al igual que en PPO, mostramos las recompensas episódicas guardadas durante el entrenamiento y una media móvil de hasta 50 episodios.

In [ ]:
# Extraemos las recompensas de los episodios que DQN registró durante el aprendizaje.
recompensas_episodios_dqn = [float(info['r']) for info in model_dqn.ep_info_buffer]
if not recompensas_episodios_dqn:
    raise RuntimeError('DQN no registró episodios; no es posible graficar la curva.')

serie_dqn = pd.Series(recompensas_episodios_dqn, name='Recompensa por episodio')
media_movil_dqn = serie_dqn.rolling(window=50, min_periods=1).mean()

# Graficamos y guardamos la curva de aprendizaje de DQN.
fig, ax = plt.subplots()
ax.plot(serie_dqn.index + 1, serie_dqn, alpha=0.45, label='Recompensa por episodio')
ax.plot(media_movil_dqn.index + 1, media_movil_dqn, linewidth=2, label='Media móvil (50 episodios)')
ax.set(title='Curva de aprendizaje — DQN', xlabel='Episodio', ylabel='Recompensa')
ax.legend()
ax.grid(alpha=0.25)
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'dqn_learning_curve.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Gráfico guardado en: {FIGURES_DIR / "dqn_learning_curve.png"}')

## Tabla comparativa

Comparamos los 3 agentes: aleatorio, PPO y DQN. Todas las métricas se calculan sobre 10 episodios de evaluación; por ello, la comparación refleja esta muestra y puede variar con otras semillas.

In [ ]:
# Resumimos cada política con estadísticas de sus recompensas de evaluación.
def resumir_recompensas(nombre, recompensas):
    """Construye una fila de métricas para una lista de recompensas episódicas."""
    return {
        'agente': nombre,
        'recompensa_promedio': float(np.mean(recompensas)),
        'recompensa_max': float(np.max(recompensas)),
        'recompensa_min': float(np.min(recompensas)),
        'desviacion_std': float(np.std(recompensas)),
    }


resultados = [
    resumir_recompensas('Aleatorio', recompensas_aleatorio),
    resumir_recompensas('PPO', recompensas_ppo),
    resumir_recompensas('DQN', recompensas_dqn),
]
df_comparacion = pd.DataFrame(resultados)
print(df_comparacion.to_string(index=False, float_format=lambda valor: f'{valor:.2f}'))

# Exportamos la tabla de resultados para su análisis posterior.
ruta_comparacion = REPORTS_DIR / 'comparacion_rl.csv'
df_comparacion.to_csv(ruta_comparacion, index=False)
print(f'Comparación guardada en: {ruta_comparacion}')

## Gráfico comparativo

Un gráfico de barras que muestre la recompensa promedio de cada agente permite comparar de un vistazo la línea base aleatoria y los agentes entrenados. Las barras no muestran por sí solas la variabilidad; para eso consultamos también la desviación estándar de la tabla.

In [ ]:
# Graficamos la recompensa promedio observada en los episodios de evaluación.
fig, ax = plt.subplots()
ax.bar(df_comparacion['agente'], df_comparacion['recompensa_promedio'], color=['gray', 'tab:blue', 'tab:orange'])
ax.set(title='Recompensa promedio por agente', xlabel='Agente', ylabel='Recompensa promedio')
ax.grid(axis='y', alpha=0.25)
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'comparacion_agentes.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Gráfico guardado en: {FIGURES_DIR / "comparacion_agentes.png"}')

## Comparación con controlador clásico

En robótica, el RL compite con controladores clásicos. Vamos a implementar un controlador simple para CartPole y comparar. Este controlador basado en reglas usa el signo del ángulo del poste: si el ángulo es positivo, mover a la derecha; si es negativo, mover a la izquierda.

Es una regla intuitiva que no necesita entrenamiento, pero no considera explícitamente la velocidad angular ni la velocidad del carro. La evaluación nos permitirá observar qué tan competitiva resulta en esta tarea concreta.

In [ ]:
def controlador_clasico(obs):
    """Elige la dirección de empuje según el signo del ángulo del poste."""
    return 1 if obs[2] > 0 else 0


def evaluar_controlador_clasico(env, n_episodios=10):
    """Evalúa el controlador por reglas y devuelve recompensas por episodio."""
    recompensas = []

    for episodio in range(n_episodios):
        observacion, info = env.reset(seed=2000 + episodio)
        recompensa_total = 0.0
        terminado = False
        truncado = False

        while not (terminado or truncado):
            accion = controlador_clasico(observacion)
            observacion, recompensa, terminado, truncado, info = env.step(accion)
            recompensa_total += recompensa

        recompensas.append(recompensa_total)

    return recompensas


# Ejecutamos y resumimos la política clásica en 10 episodios.
recompensas_clasico = evaluar_controlador_clasico(env, n_episodios=10)
print(f'Clásico — promedio: {np.mean(recompensas_clasico):.2f}')
print(f'Clásico — máximo: {np.max(recompensas_clasico):.2f}')
print(f'Clásico — mínimo: {np.min(recompensas_clasico):.2f}')

In [ ]:
# Añadimos el controlador clásico a la tabla y actualizamos el archivo CSV.
fila_clasico = pd.DataFrame([resumir_recompensas('Clásico', recompensas_clasico)])
df_comparacion = pd.concat([df_comparacion, fila_clasico], ignore_index=True)
print(df_comparacion.to_string(index=False, float_format=lambda valor: f'{valor:.2f}'))
df_comparacion.to_csv(ruta_comparacion, index=False)
print(f'Tabla actualizada y guardada en: {ruta_comparacion}')

## Análisis

¿Cuál agente tiene mejor recompensa? ¿PPO o DQN? ¿El controlador clásico es competitivo? Usa la tabla para responder, teniendo en cuenta que entrenamos con un presupuesto acotado y evaluamos solo 10 episodios.

En CartPole, un controlador simple puede funcionar muy bien. El RL brilla cuando el entorno es complejo o cambia, cuando las reglas no son fáciles de diseñar manualmente o cuando se busca optimizar el comportamiento a partir de la experiencia. Ninguna métrica de una sola ejecución basta para afirmar que un algoritmo es universalmente mejor.

## Cuándo usar RL vs. clásico

| Criterio | Control clásico | Aprendizaje por Refuerzo |
|---|---|---|
| Requiere modelo | Suele requerir un modelo del sistema o conocimiento experto | Puede aprender por interacción, aunque necesita definir entorno y recompensa |
| Requiere entrenamiento | No siempre; puede diseñarse directamente | Sí; requiere interacciones y cómputo |
| Adaptabilidad | Depende de cómo se diseñe y ajuste el controlador | Puede adaptarse mediante entrenamiento o reentrenamiento |
| Interpretabilidad | Frecuentemente alta en controladores sencillos | Depende del algoritmo; las políticas neuronales pueden ser difíciles de explicar |
| Garantías | Se pueden analizar formalmente para ciertos sistemas y supuestos | Las garantías dependen del método y del entorno; no se deben asumir automáticamente |

> **RL no reemplaza a los clásicos. Los complementa.** La selección debe considerar seguridad, datos disponibles, costo de entrenamiento, validación y garantías necesarias.

## Conclusiones

1. **PPO vs. DQN:** PPO admite acciones discretas y continuas; DQN se enfoca en acciones discretas y reutiliza experiencia.
2. **Stable-Baselines3:** ofrece una API uniforme para probar implementaciones conocidas de algoritmos de RL.
3. **Entrenamiento:** el agente aprende mediante interacciones; el presupuesto de pasos y los hiperparámetros influyen en el resultado.
4. **Evaluación:** comparar episodios de evaluación y curvas de aprendizaje permite entender el desempeño y su variabilidad.
5. **Controlador clásico:** una regla simple puede ser una línea base competitiva; RL es una alternativa complementaria cuando las reglas no bastan.

**Próxima clase:** cierre del Bloque 2, optimización de rendimiento y conexión con el Bloque 3.

In [ ]:
# Cerramos el entorno para liberar sus recursos antes de finalizar.
env.close()
print('Clase 12 completada. Revisa reports/figures/ y reports/comparacion_rl.csv')